# 02 - Análisis exploratorio y factores asociados

Corresponde a la etapa de **Comprensión de los datos** del proceso CRISP-DM y al **objetivo específico 1**: identificar los factores asociados al diagnóstico autorreportado de Diabetes Mellitus en adultos de 30 a 69 años de Lima Metropolitana.

Sigue la sección 3.1 de `docs/protocolo_analisis.md`, fijada antes de ejecutar este análisis:

- Descriptivos ponderados con el factor de expansión de la ENDES; pruebas bivariadas sin ponderar.
- OR crudos y ajustados por regresión logística multivariable (casos completos).
- **Criterio de HE1:** edad, IMC, perímetro abdominal e hipertensión diagnosticada se consideran asociados si su OR ajustado tiene p < 0.05.
- Análisis de sensibilidad por la colinealidad entre IMC y perímetro abdominal.

**Solo se usan los datos 2021–2024.**

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

from preparacion import DESENLACE, PREDICTORES_COMPLETO, cargar_desarrollo

SALIDA = Path('../resultados/02_exploracion')
SALIDA.mkdir(parents=True, exist_ok=True)
pd.set_option('display.float_format', '{:.3f}'.format)

# etiqueta, tipo y unidades por las que se expresa el OR
VARIABLES = {
    'edad': ('Edad (años)', 'continua', 10),
    'hombre': ('Sexo masculino', 'binaria', 1),
    'imc': ('IMC (kg/m²)', 'continua', 5),
    'cintura': ('Perímetro abdominal (cm)', 'continua', 10),
    'hta': ('Hipertensión diagnosticada', 'binaria', 1),
    'tabaco': ('Fumó en los últimos 12 meses', 'binaria', 1),
    'alcohol': ('Bebió alcohol en los últimos 12 meses', 'binaria', 1),
    'frutas_dias': ('Días con frutas (última semana)', 'continua', 1),
    'verduras_dias': ('Días con ensalada de verduras (última semana)', 'continua', 1),
    'pas': ('Presión sistólica medida (mmHg)', 'continua', 10),
    'pad': ('Presión diastólica medida (mmHg)', 'continua', 10),
    'educacion': ('Nivel educativo (0-5)', 'ordinal', 1),
    'seguro': ('Tiene seguro de salud', 'binaria', 1),
    'riqueza': ('Quintil de riqueza (1-5)', 'ordinal', 1),
}
FACTORES_HE1 = ['edad', 'imc', 'cintura', 'hta']

d = cargar_desarrollo()
print(f'Desarrollo 2021-2024: {len(d)} registros, {int(d[DESENLACE].sum())} con diagnóstico autorreportado')

Desarrollo 2021-2024: 8909 registros, 562 con diagnóstico autorreportado


## Prevalencia del diagnóstico autorreportado por año

La prevalencia ponderada usa el factor de expansión `PESO15_AMAS`; la cruda corresponde a la muestra sin ponderar.

In [2]:
def media_ponderada(x, w):
    ok = x.notna() & w.notna()
    return float(np.average(x[ok], weights=w[ok]))


prevalencia = pd.DataFrame([
    {
        'anio': anio,
        'registros': len(g),
        'casos': int(g[DESENLACE].sum()),
        'prevalencia_cruda_%': 100 * g[DESENLACE].mean(),
        'prevalencia_ponderada_%': 100 * media_ponderada(g[DESENLACE], g['peso_muestral']),
    }
    for anio, g in d.groupby('anio')
])
prevalencia.to_csv(SALIDA / 'prevalencia_anual.csv', index=False)
prevalencia

,anio,registros,casos,prevalencia_cruda_%,prevalencia_ponderada_%
0,2021,2357,142,6.025,7.843
1,2022,2256,136,6.028,7.396
2,2023,2135,155,7.260,9.279
3,2024,2161,129,5.969,7.645


## Características según diagnóstico autorreportado

Porcentajes y medias ponderados. Valor p de pruebas sin ponderar: chi-cuadrado para variables binarias y Mann-Whitney para continuas y ordinales.

In [3]:
grupos = {'Total': d, 'Con DM': d[d[DESENLACE] == 1], 'Sin DM': d[d[DESENLACE] == 0]}
filas = []
for var, (etiqueta, tipo, _) in VARIABLES.items():
    fila = {'variable': etiqueta, 'medida': '% ponderado' if tipo == 'binaria' else 'media ponderada'}
    for nombre, g in grupos.items():
        valor = media_ponderada(g[var], g['peso_muestral'])
        fila[nombre] = 100 * valor if tipo == 'binaria' else valor
    if tipo == 'binaria':
        fila['p'] = stats.chi2_contingency(pd.crosstab(d[var], d[DESENLACE]))[1]
    else:
        fila['p'] = stats.mannwhitneyu(grupos['Con DM'][var].dropna(), grupos['Sin DM'][var].dropna()).pvalue
    filas.append(fila)

caracteristicas = pd.DataFrame(filas)
caracteristicas.to_csv(SALIDA / 'caracteristicas.csv', index=False)
caracteristicas

,variable,medida,Total,Con DM,Sin DM,p
0,Edad (años),media ponderada,46.344,54.691,45.613,0.000
1,Sexo masculino,% ponderado,47.717,43.745,48.065,0.029
2,IMC (kg/m²),media ponderada,29.191,30.625,29.065,0.000
3,Perímetro abdominal (cm),media ponderada,97.221,102.047,96.799,0.000
4,Hipertensión diagnosticada,% ponderado,11.857,37.958,9.573,0.000
5,Fumó en los últimos 12 meses,% ponderado,16.648,14.762,16.813,0.330
6,Bebió alcohol en los últimos 12 meses,% ponderado,76.784,67.545,77.593,0.000
7,Días con frutas (última semana),media ponderada,4.023,4.158,4.011,0.406
8,Días con ensalada de verduras (última semana),media ponderada,2.832,3.133,2.806,0.004
9,Presión sistólica medida (mmHg),media ponderada,124.241,131.732,123.585,0.000


## Datos faltantes

In [4]:
faltantes = pd.DataFrame({
    'variable': [VARIABLES[v][0] for v in VARIABLES],
    'faltantes': [int(d[v].isna().sum()) for v in VARIABLES],
    '%': [100 * d[v].isna().mean() for v in VARIABLES],
})
faltantes.to_csv(SALIDA / 'faltantes.csv', index=False)
faltantes[faltantes['faltantes'] > 0]

,variable,faltantes,%
4,Hipertensión diagnosticada,7,0.079
5,Fumó en los últimos 12 meses,1,0.011
6,Bebió alcohol en los últimos 12 meses,1,0.011
7,Días con frutas (última semana),2,0.022
8,Días con ensalada de verduras (última semana),1,0.011
9,Presión sistólica medida (mmHg),1,0.011
10,Presión diastólica medida (mmHg),1,0.011
11,Nivel educativo (0-5),56,0.629


## Odds ratios crudos y ajustados

Regresión logística. Los OR ajustados provienen de un único modelo multivariable con todos los predictores del modelo completo, en casos completos. Las variables continuas se expresan por unidades interpretables (por ejemplo, edad por cada 10 años).

In [5]:
def escalar(datos, variables):
    return pd.DataFrame({v: datos[v] / VARIABLES[v][2] for v in variables}, index=datos.index)


def logit(datos, variables):
    X = sm.add_constant(escalar(datos, variables))
    return sm.Logit(datos[DESENLACE], X).fit(disp=False)


def tabla_or(modelo, variables):
    ic = modelo.conf_int()
    return pd.DataFrame({
        'OR': np.exp(modelo.params[variables]),
        'IC95_inf': np.exp(ic.loc[variables, 0]),
        'IC95_sup': np.exp(ic.loc[variables, 1]),
        'p': modelo.pvalues[variables],
    })


def etiqueta_or(var):
    etiqueta, tipo, escala = VARIABLES[var]
    return f'{etiqueta}, por cada {escala}' if tipo == 'continua' and escala != 1 else etiqueta


completos = d.dropna(subset=PREDICTORES_COMPLETO)
print(f'Casos completos para el modelo multivariable: {len(completos)} de {len(d)}')

crudos = pd.concat([tabla_or(logit(d.dropna(subset=[v]), [v]), [v]) for v in PREDICTORES_COMPLETO])
ajustados = tabla_or(logit(completos, PREDICTORES_COMPLETO), PREDICTORES_COMPLETO)
odds_ratios = crudos.add_suffix('_crudo').join(ajustados.add_suffix('_ajustado'))
odds_ratios.insert(0, 'variable', [etiqueta_or(v) for v in odds_ratios.index])
odds_ratios.to_csv(SALIDA / 'odds_ratios.csv', index=False)
odds_ratios

Casos completos para el modelo multivariable: 8840 de 8909


,variable,OR_crudo,IC95_inf_crudo,IC95_sup_crudo,p_crudo,OR_ajustado,IC95_inf_ajustado,IC95_sup_ajustado,p_ajustado
edad,"Edad (años), por cada 10",2.158,2.000,2.328,0.000,1.793,1.630,1.972,0.000
hombre,Sexo masculino,0.821,0.690,0.977,0.026,0.806,0.649,1.000,0.050
imc,"IMC (kg/m²), por cada 5",1.305,1.210,1.408,0.000,0.702,0.564,0.873,0.001
cintura,"Perímetro abdominal (cm), por cada 10",1.436,1.340,1.539,0.000,1.695,1.386,2.073,0.000
hta,Hipertensión diagnosticada,6.319,5.229,7.636,0.000,3.010,2.413,3.756,0.000
tabaco,Fumó en los últimos 12 meses,0.885,0.703,1.116,0.302,1.344,1.037,1.742,0.025
alcohol,Bebió alcohol en los últimos 12 meses,0.593,0.494,0.713,0.000,0.805,0.655,0.991,0.041
frutas_dias,Días con frutas (última semana),1.014,0.980,1.049,0.433,1.015,0.976,1.055,0.463
verduras_dias,Días con ensalada de verduras (última semana),1.058,1.021,1.097,0.002,1.072,1.031,1.116,0.001
pas,"Presión sistólica medida (mmHg), por cada 10",1.261,1.211,1.312,0.000,1.000,0.931,1.074,0.999


## Colinealidad entre IMC y perímetro abdominal

Se reportan la correlación y los factores de inflación de la varianza (VIF). Como análisis de sensibilidad, se reestima el modelo ajustado excluyendo, por separado, cada una de las dos variables. Según el protocolo, este análisis **no cambia el criterio de HE1**.

In [6]:
print(f"Correlación IMC - perímetro abdominal: {completos['imc'].corr(completos['cintura']):.2f}")

Xv = sm.add_constant(completos[PREDICTORES_COMPLETO])
vif = pd.DataFrame({
    'variable': [VARIABLES[v][0] for v in PREDICTORES_COMPLETO],
    'VIF': [variance_inflation_factor(Xv.values, i + 1) for i in range(len(PREDICTORES_COMPLETO))],
})
vif.to_csv(SALIDA / 'vif.csv', index=False)
vif

Correlación IMC - perímetro abdominal: 0.90


,variable,VIF
0,Edad (años),1.447
1,Sexo masculino,1.448
2,IMC (kg/m²),6.550
3,Perímetro abdominal (cm),6.713
4,Hipertensión diagnosticada,1.181
5,Fumó en los últimos 12 meses,1.126
6,Bebió alcohol en los últimos 12 meses,1.090
7,Días con frutas (última semana),1.134
8,Días con ensalada de verduras (última semana),1.099
9,Presión sistólica medida (mmHg),3.049


In [7]:
sensibilidad = []
for excluida in ['cintura', 'imc']:
    variables = [v for v in PREDICTORES_COMPLETO if v != excluida]
    t = tabla_or(logit(completos, variables), [v for v in FACTORES_HE1 if v != excluida])
    t.insert(0, 'modelo', f'sin {VARIABLES[excluida][0]}')
    sensibilidad.append(t)
sensibilidad = pd.concat(sensibilidad)
sensibilidad.insert(1, 'variable', [etiqueta_or(v) for v in sensibilidad.index])
sensibilidad.to_csv(SALIDA / 'sensibilidad_imc_cintura.csv', index=False)
sensibilidad

,modelo,variable,OR,IC95_inf,IC95_sup,p
edad,sin Perímetro abdominal (cm),"Edad (años), por cada 10",1.893,1.725,2.077,0.000
imc,sin Perímetro abdominal (cm),"IMC (kg/m²), por cada 5",1.180,1.083,1.285,0.000
hta,sin Perímetro abdominal (cm),Hipertensión diagnosticada,3.053,2.449,3.807,0.000
edad,sin IMC (kg/m²),"Edad (años), por cada 10",1.857,1.693,2.038,0.000
cintura,sin IMC (kg/m²),"Perímetro abdominal (cm), por cada 10",1.252,1.157,1.356,0.000
hta,sin IMC (kg/m²),Hipertensión diagnosticada,2.989,2.396,3.728,0.000


## Contrastación de HE1

In [8]:
he1 = ajustados.loc[FACTORES_HE1].copy()
he1.insert(0, 'variable', [etiqueta_or(v) for v in FACTORES_HE1])
he1['asociado (p < 0.05)'] = np.where(he1['p'] < 0.05, 'Sí', 'No')
he1['dirección'] = np.where(he1['OR'] > 1, 'directa', 'inversa')
he1.to_csv(SALIDA / 'he1.csv', index=False)
he1

,variable,OR,IC95_inf,IC95_sup,p,asociado (p < 0.05),dirección
edad,"Edad (años), por cada 10",1.793,1.630,1.972,0.000,Sí,directa
imc,"IMC (kg/m²), por cada 5",0.702,0.564,0.873,0.001,Sí,inversa
cintura,"Perímetro abdominal (cm), por cada 10",1.695,1.386,2.073,0.000,Sí,directa
hta,Hipertensión diagnosticada,3.010,2.413,3.756,0.000,Sí,directa


## Interpretación

**HE1.** Los cuatro factores cumplen el criterio prefijado (p < 0.05 en el modelo ajustado). La edad, el perímetro abdominal y la hipertensión diagnosticada muestran una asociación directa.

**IMC.** En el modelo ajustado, el IMC presenta una asociación **inversa**. No debe interpretarse como un efecto protector: IMC y perímetro abdominal están fuertemente correlacionados (r ≈ 0.90; VIF > 6), de modo que el OR del IMC se interpreta como el efecto de un mayor IMC *a igual perímetro abdominal*, es decir, una menor proporción de grasa abdominal. Al excluir el perímetro abdominal, la asociación del IMC es directa. Este resultado es coherente con el mayor peso de la grasa abdominal en el riesgo metabólico.

**Otros hallazgos compatibles con las limitaciones del desenlace autorreportado** (plan de tesis, 4.13):

- El consumo de alcohol es menos frecuente y el consumo de ensalada de verduras más frecuente entre quienes reportan diagnóstico, lo que sugiere **causalidad inversa** (cambios de hábitos tras el diagnóstico).
- Tener seguro de salud se asocia con reportar diagnóstico, lo que refleja probablemente un **mayor acceso al diagnóstico** más que un factor de riesgo.

Estos resultados se retoman en la discusión de la tesis.